# actinia DOP example

This notebook show a small example how to process a North Rhine-Westphalian Digital Orthophoto  [NW DOP](www.opengeodata.nrw.de/produkte/geobasis/lusat/akt/dop/dop_jp2_f10/) (data license: [Datenlizenz Deutschland - Zero - Version 2.0](https://www.govdata.de/dl-de/zero-2-0)) in actinia.
It calculates the NDVI and filters by thresholds the green and water pixels. 

---

## actinia documentation

* [Swagger](https://actinia.mundialis.de/api/v3/swagger.json)
* [Stable actinia API v3 docs](https://redocly.github.io/redoc/?url=https://actinia.mundialis.de/api/v3/swagger.json)

---

## Requirements

### Software & Modules

This tutorial assumes your are comfortable with the [Python](https://python.org) programming language. Familiarity with basic REST API concepts and usage is also assumed.

Python modules used in this tutorial are:
* [requests](http://docs.python-requests.org/)
* [json](https://docs.python.org/3/library/json.html)


### Actinia API user and password

This demo requires credentials for authentication set below in **Preparation** as a variable. Another actinia instance might require different credentials.

### Helper Modules and Functions

Before interacting with the actinia server using Python, we will import required packages an set up a helper function to print formatted JSON using json.

***Note:*** *You may need to install two helpful browser plugins called **RESTman** and **JSON Formatter** that format JSON and makes it easier to read:*

* [RESTman extension](https://chrome.google.com/webstore/detail/restman/ihgpcfpkpmdcghlnaofdmjkoemnlijdi)
* [JSON Formatter](https://chrome.google.com/webstore/detail/json-formatter/bcjindcccaagfpapjjmafapmmgkkhgoa)

## Preparation


In [1]:
# first, let's import the required packages.

import json

import requests
from requests.auth import HTTPBasicAuth


To simplify our life in terms of server communication we store the credentials and REST server URL in  variables.

In [2]:
# variables to set the actinia host, version, and user

actinia_baseurl = "http://localhost:8088"
actinia_url = f"{actinia_baseurl}/api/v3"
actinia_auth = HTTPBasicAuth('actinia-gdi', 'actinia-gdi')

In [3]:
# helper function to print formatted JSON using the json module

def print_as_json(data):
    print(json.dumps(data, indent=2))

# helper function to verify a request
def verify_request(request, success_code=200):
    if request.status_code != success_code:
        print(f"ERROR: actinia processing failed with status code {request.status_code}!")
        print("See errors below:")
        print_as_json(request.json())
        request_url = request.json()["urls"]["status"]
        requests.delete(url=request_url, auth=actinia_auth)
        raise Exception(f"The resource <{request_url}> has been terminated.")

In [4]:
# verify actinia connection

request_url = f"{actinia_url}/version"
print("actinia GET request:")
print(request_url)

request = requests.get(url=request_url, auth=actinia_auth)

# check if anything went wrong
verify_request(request, 200)

# get a json-encoded content of the response
jsonResponse = request.json()

# print formatted JSON
print("actinia version:")
print_as_json(jsonResponse)

actinia GET request:
http://localhost:8088/api/v3/version
actinia version:
{
  "api_version": "3.8.0",
  "grass_version": {
    "build_date": "2026-01-13",
    "build_off_t_size": "8",
    "build_platform": "x86_64-pc-linux-musl",
    "date": "2025",
    "gdal": "3.11.5",
    "geos": "3.14.1",
    "libgis_date": "2026-01-02T21:25:59+00:00",
    "libgis_revision": "83362df5af",
    "proj": "9.7.1",
    "revision": "d2ee7c143d",
    "sqlite": "3.51.1",
    "version": "8.5.0dev"
  },
  "plugin_versions": {
    "actinia_metadata_plugin": "1.0.4",
    "actinia_module_plugin": "2.5.1",
    "actinia_satellite_plugin": "0.3.0",
    "actinia_stac_plugin": "0.4.0",
    "actinia_statistic_plugin": "0.4.0"
  },
  "plugins": "actinia_metadata_plugin,actinia_module_plugin,actinia_satellite_plugin,actinia_stac_plugin,actinia_statistic_plugin",
  "python_version": "3.12.12 (main, Oct 11 2025, 01:16:26) [GCC 15.2.0]",
  "running_since": "n/a",
  "version": "7.0.2"
}


## GRASS-Module

GRASS GIS has many modules which can be used in actinia.
A list of all GRASS-modules can be requested by:

In [5]:
# make a GET request to the actinia API to get all GRASS modules
request_url = f"{actinia_url}/grass_modules"
print("actinia GET request:")
print(request_url)

request = requests.get(url=request_url, auth=actinia_auth)

# check if anything went wrong
verify_request(request, 200)

# get a json-encoded content of the response
jsonResponse = request.json()

# print formatted JSON
print(f"{len(jsonResponse['processes'])} available GRASS-modules:")
print_as_json(jsonResponse)

actinia GET request:
http://localhost:8088/api/v3/grass_modules
562 available GRASS-modules:
{
  "processes": [
    {
      "categories": [
        "background",
        "display",
        "erase",
        "fill",
        "graphics",
        "grass-module"
      ],
      "description": "Fills the graphics display frame with user defined color.",
      "id": "d.background"
    },
    {
      "categories": [
        "cartography",
        "display",
        "grass-module"
      ],
      "description": "Displays a barscale on the graphics monitor.",
      "id": "d.barscale"
    },
    {
      "categories": [
        "colors",
        "display",
        "grass-module",
        "settings"
      ],
      "description": "Outputs a list of all available display colors.",
      "id": "d.colorlist"
    },
    {
      "categories": [
        "color table",
        "display",
        "grass-module",
        "raster"
      ],
      "description": "Displays the color table associated with a raster m

Description of one module:

In [6]:
# make a GET request to the actinia API to get all GRASS modules
request_url = f"{actinia_url}/grass_modules/r.info"
print("actinia GET request:")
print(request_url)
request = requests.get(url=request_url, auth=actinia_auth)
verify_request(request, 200)
jsonResponse = request.json()
print(f"r.info description in actinia:")
print_as_json(jsonResponse)

actinia GET request:
http://localhost:8088/api/v3/grass_modules/r.info
r.info description in actinia:
{
  "categories": [
    "extent",
    "grass-module",
    "history",
    "metadata",
    "raster"
  ],
  "description": "Outputs basic information about a raster map.",
  "id": "r.info",
  "parameters": [
    {
      "description": "Name of raster map. ",
      "name": "map",
      "optional": false,
      "schema": {
        "subtype": "cell",
        "type": "string"
      }
    },
    {
      "description": "Output format. ",
      "name": "format",
      "optional": true,
      "schema": {
        "enum": [
          "plain",
          "shell",
          "json"
        ],
        "type": "string"
      }
    },
    {
      "default": "False",
      "description": "Print raster array information. ",
      "name": "g",
      "optional": true,
      "schema": {
        "type": "boolean"
      }
    },
    {
      "default": "False",
      "description": "Print range. ",
      "name": 

## actinia Process chain

Process chain imports DOP tile and make small analysis.
It contains:
* data import
* region setting (important in GRASS GIS)
* data analysis
* returning NDVI infos
* exporting NDVI and water and green pixels as rasters

```json
{
    "list": [
        {
            "id": "import_dop_tile",
            "module": "importer",
            "inputs": [
                {
                    "import_descr": {
                        "source": "https://www.opengeodata.nrw.de/produkte/geobasis/lusat/akt/dop/dop_jp2_f10/dop10rgbi_32_366_5621_1_nw_2025.jp2",
                        "type": "raster"
                    },
                    "param": "map",
                    "value": "dop"
                }
            ]
        },
        {
            "id": "set_region_to_dop",
            "module": "g.region",
            "inputs": [
                {
                    "param": "raster",
                    "value": "dop.1"
                }
            ]
        },
        {
            "id": "calculate_ndvi",
            "module": "r.mapcalc",
            "inputs": [
                {
                    "param": "expression",
                    "value": "ndvi = (dop.4 - dop.1) / float((dop.4 + dop.1))"
                }
            ]
        },
        {
            "id": "ndvi_color",
            "module": "r.colors",
            "inputs": [
                {
                    "param": "map",
                    "value": "ndvi"
                },
                {
                    "param": "color",
                    "value": "ndvi"
                }
            ]
        },
        {
            "id": "ndvi_info",
            "module": "r.info",
            "inputs": [
                {
                    "param": "map",
                    "value": "ndvi"
                }
            ],
            "flags": "g",
            "stdout": {"id": "ndvi_info", "format": "kv", "delimiter": "="}
        },
        {
            "id": "green_pixels",
            "module": "r.mapcalc",
            "inputs": [
                {
                    "param": "expression",
                    "value": "green_px = if(ndvi > 0.3, 1, null() )"
                }
            ]
        },
        {
            "id": "water_pixels",
            "module": "r.mapcalc",
            "inputs": [
                {
                    "param": "expression",
                    "value": "water_px = if(ndvi < -0.3, 1, null() )"
                }
            ]
        },
        {
            "id": "export_data",
            "module": "exporter",
            "outputs": [
                {
                    "export": {
                        "type": "raster",
                        "format": "GTiff"
                    },
                    "param": "map",
                    "value": "ndvi"
                },
                {
                    "export": {
                        "type": "raster",
                        "format": "GTiff"
                    },
                    "param": "map",
                    "value": "green_px"
                },
                {
                    "export": {
                        "type": "raster",
                        "format": "GTiff"
                    },
                    "param": "map",
                    "value": "water_px"
                }
            ]
        }
    ],
    "version": "1"
}
```

It is important to start processes in the right GRASS GIS project.
List all projects that are available in the actinia persistent database:

In [7]:
# make a GET request to the actinia data API
request_url = f"{actinia_url}/projects"
print("actinia GET request:")
print(request_url)

request = requests.get(url=request_url, auth=actinia_auth)
verify_request(request, 200)
jsonResponse = request.json()

# print formatted JSON
print("Available projects:")
print_as_json(jsonResponse)

actinia GET request:
http://localhost:8088/api/v3/projects
Available projects:
{
  "projects": [
    "nc_spm_08"
  ],
  "status": "success"
}


If there is no suitable project, then a new one can simply be created:

In [8]:
# make a POST request to the actinia API
request_url = f"{actinia_url}/projects/dop_example"
print("actinia POST request:")
print(request_url)

project_cfg = {"epsg":"25832"}
request = requests.post(url=request_url, auth=actinia_auth, json=project_cfg)
# check if anything went wrong
verify_request(request, 200)
# get a json-encoded content of the response
jsonResponse = request.json()
print(f"Response with status code: {request.status_code}")

actinia POST request:
http://localhost:8088/api/v3/projects/dop_example
Response with status code: 200


Do processing in new project:

In [9]:
request_url = f"{actinia_url}/locations/dop_example/processing_export"
print("actinia POST request:")
print(request_url)
process_chain = {
    "list": [
        {
            "id": "import_dop_tile",
            "module": "importer",
            "inputs": [
                {
                    "import_descr": {
                        "source": "https://www.opengeodata.nrw.de/produkte/geobasis/lusat/akt/dop/dop_jp2_f10/dop10rgbi_32_366_5621_1_nw_2025.jp2",
                        "type": "raster"
                    },
                    "param": "map",
                    "value": "dop"
                }
            ]
        },
        {
            "id": "set_region_to_dop",
            "module": "g.region",
            "inputs": [
                {
                    "param": "raster",
                    "value": "dop.1"
                }
            ]
        },
        {
            "id": "calculate_ndvi",
            "module": "r.mapcalc",
            "inputs": [
                {
                    "param": "expression",
                    "value": "ndvi = (dop.4 - dop.1) / float((dop.4 + dop.1))"
                }
            ]
        },
        {
            "id": "ndvi_color",
            "module": "r.colors",
            "inputs": [
                {
                    "param": "map",
                    "value": "ndvi"
                },
                {
                    "param": "color",
                    "value": "ndvi"
                }
            ]
        },
        {
            "id": "ndvi_info",
            "module": "r.info",
            "inputs": [
                {
                    "param": "map",
                    "value": "ndvi"
                }
            ],
            "flags": "g",
            "stdout": {"id": "ndvi_info", "format": "kv", "delimiter": "="}
        },
        {
            "id": "green_pixels",
            "module": "r.mapcalc",
            "inputs": [
                {
                    "param": "expression",
                    "value": "green_px = if(ndvi > 0.3, 1, null() )"
                }
            ]
        },
        {
            "id": "water_pixels",
            "module": "r.mapcalc",
            "inputs": [
                {
                    "param": "expression",
                    "value": "water_px = if(ndvi < -0.3, 1, null() )"
                }
            ]
        },
        {
            "id": "export_data",
            "module": "exporter",
            "outputs": [
                {
                    "export": {
                        "type": "raster",
                        "format": "GTiff"
                    },
                    "param": "map",
                    "value": "ndvi"
                },
                {
                    "export": {
                        "type": "raster",
                        "format": "GTiff"
                    },
                    "param": "map",
                    "value": "green_px"
                },
                {
                    "export": {
                        "type": "raster",
                        "format": "GTiff"
                    },
                    "param": "map",
                    "value": "water_px"
                }
            ]
        }
    ],
    "version": "1"
}
request = requests.post(url=request_url, auth=actinia_auth, json=process_chain)
verify_request(request, 200)

# get a json-encoded content of the response
jsonResponse = request.json()

# print formatted JSON
print(f"Response with status code: {request.status_code}")
print_as_json(jsonResponse)

actinia POST request:
http://localhost:8088/api/v3/locations/dop_example/processing_export
Response with status code: 200
{
  "accept_datetime": "2026-01-23 08:33:09.033025",
  "accept_timestamp": 1769157189.0330224,
  "api_info": {
    "endpoint": "gdiasyncephemeralexportresource",
    "method": "POST",
    "path": "/api/v3/locations/dop_example/processing_export",
    "request_url": "http://localhost:8088/api/v3/locations/dop_example/processing_export"
  },
  "datetime": "2026-01-23 08:33:09.694996",
  "http_code": 200,
  "message": "Resource accepted",
  "process_chain_list": [],
  "process_results": {},
  "queue": "local",
  "resource_id": "resource_id-befcc396-ae7f-45f2-965c-21bb3578789b",
  "status": "accepted",
  "time_delta": 0.6619822978973389,
  "timestamp": 1769157189.6949944,
  "urls": {
    "resources": [],
    "status": "http://localhost:8088/api/v3/resources/actinia-gdi/resource_id-befcc396-ae7f-45f2-965c-21bb3578789b"
  },
  "user_id": "actinia-gdi"
}


The response provides the status URL that must be polled to receive the
finished response:

Poll the status of the asynchronous API call by polling the status URL.
Be aware that you have to use the current status url as the resource id will change for different API
calls.

In [10]:
# make a GET request to the actinia data API
request_url = jsonResponse["urls"]["status"]
print("actinia GET request:")
print(request_url)
print("---")

request = requests.get(url=request_url, auth=actinia_auth)

# check if anything went wrong
verify_request(request, 200)

# get a json-encoded content of the response
jsonResponse = request.json()

actinia GET request:
http://localhost:8088/api/v3/resources/actinia-gdi/resource_id-befcc396-ae7f-45f2-965c-21bb3578789b
---


Poll until there is a message **"Processing successfully finished"**

In [11]:
# continue polling until finished
while request.status_code == 200 and \
        jsonResponse["message"] != "Processing successfully finished":
    request = requests.get(url=request_url, auth=actinia_auth)
    jsonResponse = request.json()

# check if anything went wrong
verify_request(request, 200)

The result of the process chain evaluation is the following JSON
response.

The result of the stdout output parsing for each module is located in
the "process_results" section of the json response.

In [12]:
# print formatted JSON
print_as_json(jsonResponse)

{
  "accept_datetime": "2026-01-23 08:33:09.033025",
  "accept_timestamp": 1769157189.0330224,
  "api_info": {
    "endpoint": "gdiasyncephemeralexportresource",
    "method": "POST",
    "path": "/api/v3/locations/dop_example/processing_export",
    "request_url": "http://localhost:8088/api/v3/locations/dop_example/processing_export"
  },
  "datetime": "2026-01-23 08:36:04.264314",
  "http_code": 200,
  "message": "Processing successfully finished",
  "process_chain_list": [
    {
      "list": [
        {
          "id": "import_dop_tile",
          "inputs": [
            {
              "import_descr": {
                "source": "https://www.opengeodata.nrw.de/produkte/geobasis/lusat/akt/dop/dop_jp2_f10/dop10rgbi_32_366_5621_1_nw_2025.jp2",
                "type": "raster"
              },
              "param": "map",
              "value": "dop"
            }
          ],
          "module": "importer"
        },
        {
          "id": "set_region_to_dop",
          "inputs":

Process results:

In [13]:
print_as_json(jsonResponse["process_results"])

{
  "ndvi_info": {
    "cells": "100000000",
    "cols": "10000",
    "datatype": "FCELL",
    "east": "367000",
    "ewres": "0.1",
    "ncats": "0",
    "north": "5622000",
    "nsres": "0.1",
    "rows": "10000",
    "south": "5621000",
    "west": "366000"
  }
}


Exported rasters:

In [14]:
print_as_json(jsonResponse["urls"])

{
  "resources": [
    "http://localhost:8088/api/v3/resources/actinia-gdi/resource_id-befcc396-ae7f-45f2-965c-21bb3578789b/ndvi.tif",
    "http://localhost:8088/api/v3/resources/actinia-gdi/resource_id-befcc396-ae7f-45f2-965c-21bb3578789b/green_px.tif",
    "http://localhost:8088/api/v3/resources/actinia-gdi/resource_id-befcc396-ae7f-45f2-965c-21bb3578789b/water_px.tif"
  ],
  "status": "http://localhost:8088/api/v3/resources/actinia-gdi/resource_id-befcc396-ae7f-45f2-965c-21bb3578789b"
}


## actinia-Modules

To simplify things a little, there are actinia-modules. These are templates for a process chain in which only a specific part needs to be customized, such as the dop tile in this example.

In [ ]:
# make a GET request to the actinia API to get all actinia modules
request_url = f"{actinia_url}/actinia_modules"
print("actinia GET request:")
print(request_url)
request = requests.get(url=request_url, auth=actinia_auth)
verify_request(request, 200)
jsonResponse = request.json()
print(f"{len(jsonResponse['processes'])} available actinia-modules:")
print_as_json(jsonResponse)

actinia GET request:
http://localhost:8088/api/v3/actinia_modules
13 available actinia-modules:
{
  "processes": [
    {
      "categories": [
        "actinia-module",
        "global-template"
      ],
      "description": "Example PC to add possible values",
      "id": "add_enumeration"
    },
    {
      "categories": [
        "actinia-module",
        "global-template"
      ],
      "description": "test default value in actinia-module-plugin",
      "id": "default_value"
    },
    {
      "categories": [
        "actinia-module",
        "global-template"
      ],
      "description": "Example DOP analysis on basis of the NDVI.",
      "id": "dop_ndvi_analysis"
    },
    {
      "categories": [
        "actinia-module",
        "global-template"
      ],
      "description": "Placeholder json file with if statement",
      "id": "if_statement"
    },
    {
      "categories": [
        "actinia-module",
        "global-template"
      ],
      "description": "Example loop wit

The process chain has been converted into actinia-modules (using format of jinja2 template) so that it can be easily applied to different DOP tiles. Example actinia-modules can be found [here](https://github.com/actinia-org/actinia-module-plugin/tree/main/config/templates/pc_templates).

**actinia-module management**: The actinia-module can be created, read, updated and deleted via HTTP REST API:
```bash
curl -u XXX -X POST -H "Content-Type: application/json" -d "${JSON}" http://localhost:8088/api/v3/actinia_modules
curl -u XXX -X GET http://localhost:8088/api/v3/actinia_modules/dop_ndvi_analysis
curl -u XXX -X PUT -H "Content-Type: application/json" -d "${JSON}" http://localhost:8088/api/v3/actinia_modules/dop_ndvi_analysis
curl -u XXX -X DELETE http://localhost:8088/api/v3/actinia_modules/dop_ndvi_analysis
```


```json
{
	"id": "dop_ndvi_analysis",
	"description": "Example DOP analysis on basis of the NDVI.",
	"template": {
        "list": [
            {
                "id": "import_dop_tile",
                "module": "importer",
                "inputs": [
                    {
                        "import_descr": {
                            "source": "{{ dop_source }}",
                            "type": "raster"
                        },
                        "param": "map",
                        "value": "dop"
                    }
                ]
            },
            {
                "id": "set_region_to_dop",
                "module": "g.region",
                "inputs": [
                    {
                        "param": "raster",
                        "value": "dop.1"
                    }
                ]
            },
            {
                "id": "calculate_ndvi",
                "module": "r.mapcalc",
                "inputs": [
                    {
                        "param": "expression",
                        "value": "ndvi = (dop.4 - dop.1) / float((dop.4 + dop.1))"
                    }
                ]
            },
            {
                "id": "ndvi_color",
                "module": "r.colors",
                "inputs": [
                    {
                        "param": "map",
                        "value": "ndvi"
                    },
                    {
                        "param": "color",
                        "value": "ndvi"
                    }
                ]
            },
            {
                "id": "ndvi_info",
                "module": "r.info",
                "inputs": [
                    {
                        "param": "map",
                        "value": "ndvi"
                    }
                ],
                "flags": "g",
                "stdout": {"id": "ndvi_info", "format": "kv", "delimiter": "="}
            },
            {
                "id": "green_pixels",
                "module": "r.mapcalc",
                "inputs": [
                    {
                        "param": "expression",
                        "value": "green_px = if(ndvi > 0.3, 1, null() )"
                    }
                ]
            },
            {
                "id": "water_pixels",
                "module": "r.mapcalc",
                "inputs": [
                    {
                        "param": "expression",
                        "value": "water_px = if(ndvi < -0.3, 1, null() )"
                    }
                ]
            },
            {
                "id": "export_data",
                "module": "exporter",
                "outputs": [
                    {
                        "export": {
                            "type": "raster",
                            "format": "GTiff"
                        },
                        "param": "map",
                        "value": "ndvi"
                    },
                    {
                        "export": {
                            "type": "raster",
                            "format": "GTiff"
                        },
                        "param": "map",
                        "value": "green_px"
                    },
                    {
                        "export": {
                            "type": "raster",
                            "format": "GTiff"
                        },
                        "param": "map",
                        "value": "water_px"
                    }
                ]
            }
        ],
        "version": "1"
    }
}
```

Get self-description of the new actinia-module:

In [ ]:
# make a GET request to the actinia API
request_url = f"{actinia_url}/actinia_modules/dop_ndvi_analysis"
print("actinia GET request:")
print(request_url)
request = requests.get(url=request_url, auth=actinia_auth)
verify_request(request, 200)
jsonResponse = request.json()
print("dop_ndvi_analysis description:")
print_as_json(jsonResponse)

actinia GET request:
http://localhost:8088/api/v3/actinia_modules/dop_ndvi_analysis
dop_ndvi_analysis description:
{
  "categories": [
    "actinia-module",
    "global-template"
  ],
  "description": "Example DOP analysis on basis of the NDVI.",
  "id": "dop_ndvi_analysis",
  "parameters": [
    {
      "description": "The input source that may be a landsat scene name, a sentinel2 scene name, a postGIS database string, or an URL that points to an accessible raster or vector file [generated from import_descr_source]",
      "name": "dop_source",
      "optional": true,
      "schema": {
        "type": "string"
      }
    }
  ],
  "projects": [],
  "returns": []
}


Using the actinia-Module:

In [17]:
request_url = f"{actinia_url}/locations/dop_example/processing_export"
print("actinia POST request:")
print(request_url)
process_chain = {
    "list": [
        {
            "id": "dop_ndvi_analysis",
            "module": "dop_ndvi_analysis",
            "inputs": [
                {
                    "param": "dop_source",
                    "value": "https://www.opengeodata.nrw.de/produkte/geobasis/lusat/akt/dop/dop_jp2_f10/dop10rgbi_32_366_5622_1_nw_2025.jp2"
                }
            ]
        }
    ],
    "version": "1"
}
request = requests.post(url=request_url, auth=actinia_auth, json=process_chain)
verify_request(request, 200)

# get a json-encoded content of the response
jsonResponse = request.json()

# print formatted JSON
print(f"Response with status code: {request.status_code}")
print_as_json(jsonResponse)

actinia POST request:
http://localhost:8088/api/v3/locations/dop_example/processing_export
Response with status code: 200
{
  "accept_datetime": "2026-01-23 08:36:05.790411",
  "accept_timestamp": 1769157365.7904088,
  "api_info": {
    "endpoint": "gdiasyncephemeralexportresource",
    "method": "POST",
    "path": "/api/v3/locations/dop_example/processing_export",
    "request_url": "http://localhost:8088/api/v3/locations/dop_example/processing_export"
  },
  "datetime": "2026-01-23 08:36:06.409829",
  "http_code": 200,
  "message": "Resource accepted",
  "process_chain_list": [],
  "process_results": {},
  "queue": "local",
  "resource_id": "resource_id-86a36050-f9c7-4776-a865-4f1cf5573256",
  "status": "accepted",
  "time_delta": 0.6194283962249756,
  "timestamp": 1769157366.409828,
  "urls": {
    "resources": [],
    "status": "http://localhost:8088/api/v3/resources/actinia-gdi/resource_id-86a36050-f9c7-4776-a865-4f1cf5573256"
  },
  "user_id": "actinia-gdi"
}


In [18]:
# make a GET request to the actinia data API
request_url = jsonResponse["urls"]["status"]
print("actinia GET request:")
print(request_url)
print("---")

request = requests.get(url=request_url, auth=actinia_auth)

# check if anything went wrong
verify_request(request, 200)

# get a json-encoded content of the response
jsonResponse = request.json()

actinia GET request:
http://localhost:8088/api/v3/resources/actinia-gdi/resource_id-86a36050-f9c7-4776-a865-4f1cf5573256
---


In [19]:
# continue polling until finished
while request.status_code == 200 and \
        jsonResponse["message"] != "Processing successfully finished":
    request = requests.get(url=request_url, auth=actinia_auth)
    jsonResponse = request.json()

# check if anything went wrong
verify_request(request, 200)

In [20]:
print_as_json(jsonResponse["process_results"])

{
  "ndvi_info": {
    "cells": "100000000",
    "cols": "10000",
    "datatype": "FCELL",
    "east": "367000",
    "ewres": "0.1",
    "ncats": "0",
    "north": "5623000",
    "nsres": "0.1",
    "rows": "10000",
    "south": "5622000",
    "west": "366000"
  }
}


In [21]:
print_as_json(jsonResponse["urls"])

{
  "resources": [
    "http://localhost:8088/api/v3/resources/actinia-gdi/resource_id-86a36050-f9c7-4776-a865-4f1cf5573256/ndvi.tif",
    "http://localhost:8088/api/v3/resources/actinia-gdi/resource_id-86a36050-f9c7-4776-a865-4f1cf5573256/green_px.tif",
    "http://localhost:8088/api/v3/resources/actinia-gdi/resource_id-86a36050-f9c7-4776-a865-4f1cf5573256/water_px.tif"
  ],
  "status": "http://localhost:8088/api/v3/resources/actinia-gdi/resource_id-86a36050-f9c7-4776-a865-4f1cf5573256"
}


**GRASS**- and **actinia**-modules are together available in an modules endpoint:

In [22]:
# make a GET request to the actinia API to get all modules
request_url = f"{actinia_url}/modules"
print("actinia GET request:")
print(request_url)
request = requests.get(url=request_url, auth=actinia_auth)
verify_request(request, 200)
jsonResponse = request.json()
print(f"{len(jsonResponse['processes'])} available actinia-modules:")
print_as_json(jsonResponse)

actinia GET request:
http://localhost:8088/api/v3/modules
575 available actinia-modules:
{
  "processes": [
    {
      "categories": [
        "background",
        "display",
        "erase",
        "fill",
        "graphics",
        "grass-module"
      ],
      "description": "Fills the graphics display frame with user defined color.",
      "id": "d.background"
    },
    {
      "categories": [
        "cartography",
        "display",
        "grass-module"
      ],
      "description": "Displays a barscale on the graphics monitor.",
      "id": "d.barscale"
    },
    {
      "categories": [
        "colors",
        "display",
        "grass-module",
        "settings"
      ],
      "description": "Outputs a list of all available display colors.",
      "id": "d.colorlist"
    },
    {
      "categories": [
        "color table",
        "display",
        "grass-module",
        "raster"
      ],
      "description": "Displays the color table associated with a raster map l